In [1]:
#%pip install merlinquantum torch numpy matplotlib

In [2]:
import math
from math import comb
import gc
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, SVG

# MerLin package import name is usually `merlin`.
try:
    import merlin as ML
except Exception as exc:
    raise ImportError(
        "Could not import MerLin. Install with: pip install merlinquantum"
    ) from exc

import perceval as pcvl

try:
    from perceval.rendering import Format as PercevalFormat
except Exception:
    PercevalFormat = pcvl.Format

print("torch", torch.__version__)
print("merlin", getattr(ML, "__version__", "version unavailable"))

torch 2.10.0+cpu
merlin 0.2.3


In [3]:
# Reproducibility and dtype
seed = 66
torch.manual_seed(seed)
np.random.seed(seed)

# MerLin's high-level layers commonly return float32 tensors.
# Keep the whole notebook in float32 to avoid dtype mismatches.
torch.set_default_dtype(torch.float32)
dtype = torch.float32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("dtype:", dtype)

device: cpu
dtype: torch.float32


In [4]:
# PDE constants
alpha = 0.1
x_min, x_max = 0.0, 1.0
t_min, t_max = 0.0, 1.0

def exact_u(x, t):
    return torch.exp(-alpha * math.pi**2 * t) * torch.sin(math.pi * x)

def sample_interior(n):
    x = torch.rand(n, 1, device=device, dtype=torch.get_default_dtype())
    t = torch.rand(n, 1, device=device, dtype=torch.get_default_dtype())
    xt = torch.cat([x, t], dim=1)
    xt.requires_grad_(True)
    return xt

def sample_initial(n):
    x = torch.rand(n, 1, device=device, dtype=torch.get_default_dtype())
    t = torch.zeros(n, 1, device=device, dtype=torch.get_default_dtype())
    xt = torch.cat([x, t], dim=1)
    xt.requires_grad_(True)
    return xt

def sample_boundary(n):
    t = torch.rand(n, 1, device=device, dtype=torch.get_default_dtype())
    left = torch.zeros(n // 2, 1, device=device, dtype=torch.get_default_dtype())
    right = torch.ones(n - n // 2, 1, device=device, dtype=torch.get_default_dtype())
    x = torch.cat([left, right], dim=0)
    t = t[: x.shape[0]]
    xt = torch.cat([x, t], dim=1)
    xt.requires_grad_(True)
    return xt

In [5]:
def add_compact_mesh(circ, n_modes, prefix, mesh_depth=None):
    """
    Add a compact nearest-neighbour interferometer mesh.

    By default, mesh_depth = n_modes. For odd mode counts, this produces
    alternating nearest-neighbour beam-splitter layers such as
        BS(0,1) -> BS(1,2) -> BS(0,1)
    when n_modes = 3.
    """
    if n_modes < 2:
        return

    if mesh_depth is None:
        mesh_depth = n_modes

    for layer in range(mesh_depth):
        start = layer % 2
        for left in range(start, n_modes - 1, 2):
            right = left + 1
            circ.add(
                (left, right),
                pcvl.BS.Rx(
                    theta=pcvl.P(f"gamma_{prefix}_l{layer}_{left}{right}"),
                    phi_tr=pcvl.P(f"phi_{prefix}_l{layer}_{left}{right}"),
                ),
            )


def balanced_photon_input_state(n_modes, n_photons):
    """Distribute the photons as evenly as possible across modes."""
    base, remainder = divmod(n_photons, n_modes)
    return [base + (1 if i < remainder else 0) for i in range(n_modes)]


def fock_output_size(n_modes, n_photons):
    """Number of Fock outcomes for n_photons in n_modes modes."""
    return comb(n_photons + n_modes - 1, n_photons)


def mesh_beam_splitter_count(n_modes, mesh_depth=None):
    """Count BS gates in one compact mesh with the same rule as add_compact_mesh."""
    if n_modes < 2:
        return 0
    if mesh_depth is None:
        mesh_depth = n_modes
    return sum(len(range(layer % 2, n_modes - 1, 2)) for layer in range(mesh_depth))


def gate_counts(n_modes, mesh_depth=None):
    """Return the analytical gate counts for the circuit used in this notebook."""
    bs_per_mesh = mesh_beam_splitter_count(n_modes, mesh_depth=mesh_depth)
    counts = {
        "BS_U1": bs_per_mesh,
        "BS_U2": bs_per_mesh,
        "BS_total": 2 * bs_per_mesh,
        "PS_input": n_modes,
        "PS_theta": n_modes,
        "PS_alpha": n_modes,
        "PS_total": 3 * n_modes,
    }
    return counts


def build_paper_inspired_circuit(n_modes, n_photons, mesh_depth=None):
    """
    Paper-inspired DV photonic circuit for arbitrary photon/mode count.

    Structure:
        U1 -> data phase encoding -> theta phases -> alpha phases -> U2

    The data and trainable phase shifts are placed before the final
    interferometer, so they can affect photon-count probabilities.
    """
    circ = pcvl.Circuit(
        n_modes,
        name=f"paper_ish_{n_photons}photons_{n_modes}modes",
    )

    # First trainable interferometer.
    add_compact_mesh(circ, n_modes, prefix="u1", mesh_depth=mesh_depth)

    # Data phase encoding after the first interferometer.
    for mode in range(n_modes):
        circ.add(mode, pcvl.PS(pcvl.P(f"input{mode}")))

    # Trainable internal phase layers.
    for mode in range(n_modes):
        circ.add(mode, pcvl.PS(pcvl.P(f"theta{mode}")))
    for mode in range(n_modes):
        circ.add(mode, pcvl.PS(pcvl.P(f"alpha{mode}")))

    # Final trainable interferometer.
    add_compact_mesh(circ, n_modes, prefix="u2", mesh_depth=mesh_depth)

    return circ

def display_circuit_inline(circ, filename):
    """
    Render a Perceval circuit to an SVG file and display it inline.

    Using an explicit display call is more reliable inside loops than relying
    on a bare `pcvl.pdisplay(...)` statement, because it forces every circuit
    to appear in the notebook output below its corresponding iteration.
    """
    render_dir = Path("rendered_circuits")
    render_dir.mkdir(exist_ok=True)
    svg_path = render_dir / filename

    try:
        pcvl.pdisplay_to_file(
            circ,
            str(svg_path),
            output_format=PercevalFormat.HTML,
            recursive=False,
        )
        display(SVG(filename=str(svg_path)))
    except Exception as exc:
        # Fallback for older Perceval versions/environments.
        print(f"SVG export failed ({exc}); falling back to pcvl.pdisplay.")
        pcvl.pdisplay(circ, recursive=False)


In [6]:
class MerlinHeatQPINNCustom(nn.Module):
    def __init__(self, n_modes, n_photons, hidden=8, mesh_depth=None):
        super().__init__()
        self.n_modes = n_modes
        self.n_photons = n_photons
        self.mesh_depth = mesh_depth

        self.feature_map = nn.Sequential(
            nn.Linear(2, hidden),
            nn.Tanh(),
            nn.Linear(hidden, n_modes),
            nn.Tanh(),
        )

        circ = build_paper_inspired_circuit(
            n_modes=n_modes,
            n_photons=n_photons,
            mesh_depth=mesh_depth,
        )
        self.circuit_to_draw = circ

        trainable_groups = ["theta", "alpha"]
        if n_modes > 1:
            trainable_groups = ["gamma", "phi"] + trainable_groups

        self.quantum = ML.QuantumLayer(
            input_size=n_modes,
            circuit=circ,
            input_parameters=["input"],
            trainable_parameters=trainable_groups,
            input_state=balanced_photon_input_state(n_modes, n_photons),
            # Full Fock basis, including bunched outcomes.
            measurement_strategy=ML.MeasurementStrategy.probs(ML.ComputationSpace.FOCK),
        )

        self.readout = nn.Sequential(
            nn.Linear(self.quantum.output_size, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 2),
        )

    def forward(self, xt):
        x = xt[:, 0:1]

        # Feature map outputs angle-like inputs in roughly [-pi, pi].
        z = math.pi * self.feature_map(xt)
        q = self.quantum(z)
        out = self.readout(q)

        q_u = out[:, 0:1]
        ux_hat = out[:, 1:2]

        # Enforce homogeneous Dirichlet boundary conditions exactly.
        u = x * (1.0 - x) * q_u
        return u, ux_hat

In [7]:
def gradients(y, x):
    return torch.autograd.grad(
        y,
        x,
        grad_outputs=torch.ones_like(y),
        create_graph=True,
        retain_graph=True,
    )[0]


def pde_and_consistency_residuals(model, xt):
    u, ux_hat = model(xt)

    grad_u = gradients(u, xt)
    u_x = grad_u[:, 0:1]
    u_t = grad_u[:, 1:2]

    grad_ux_hat = gradients(ux_hat, xt)
    ux_hat_x = grad_ux_hat[:, 0:1]

    r_f = u_t - alpha * ux_hat_x
    r_c = u_x - ux_hat
    return r_f, r_c

In [8]:
# Shared training settings for the mode/photon sweep
n_f = 64      # interior points
n_i = 64      # initial-condition points
n_b = 64      # boundary points, mostly redundant because BC is hard-coded

# Sweep over modes = 1, 3, 5, ..., 15.
# Photon count is n_photons = (n_modes + 1) / 2.
mode_list = list(range(1, 16, 2))
config_list = [(n_modes, (n_modes + 1) // 2) for n_modes in mode_list]

# The largest configuration in this sweep is large:
# 15 modes, 8 photons -> 319,770 full-Fock output probabilities.
# Keep the sweep moderate by default, then retrain the best case longer.
sweep_epochs = 150
best_retrain_epochs = 300

lr = 1e-2
lambda_f = 1.0
lambda_c = 1.0
lambda_i = 10.0
lambda_b = 1.0

hidden = 8
mesh_depth = None  # None keeps mesh_depth = n_modes.

mse = nn.MSELoss()

# Evaluation grid used for all models
nx, nt = 60, 60
x_eval = torch.linspace(0, 1, nx, device=device, dtype=dtype).reshape(-1, 1)
t_eval = torch.linspace(0, 1, nt, device=device, dtype=dtype).reshape(-1, 1)
X_eval, T_eval = torch.meshgrid(x_eval.squeeze(), t_eval.squeeze(), indexing="ij")
xt_grid = torch.stack([X_eval.reshape(-1), T_eval.reshape(-1)], dim=1)

In [9]:
# Preflight summary for the exact configurations in this sweep
header = (
    f"{'modes':>5}  {'photons':>7}  {'input_state':>24}  {'q_out':>10}  "
    f"{'BS_total':>9}  {'PS_input':>8}  {'PS_theta':>8}  {'PS_alpha':>8}  {'PS_total':>8}"
)
print(header)
print("-" * len(header))
for n_modes, n_photons in config_list:
    counts = gate_counts(n_modes, mesh_depth=mesh_depth)
    print(
        f"{n_modes:>5d}  {n_photons:>7d}  "
        f"{str(balanced_photon_input_state(n_modes, n_photons)):>24}  "
        f"{fock_output_size(n_modes, n_photons):>10d}  "
        f"{counts['BS_total']:>9d}  {counts['PS_input']:>8d}  "
        f"{counts['PS_theta']:>8d}  {counts['PS_alpha']:>8d}  "
        f"{counts['PS_total']:>8d}"
    )

modes  photons               input_state       q_out   BS_total  PS_input  PS_theta  PS_alpha  PS_total
-------------------------------------------------------------------------------------------------------
    1        1                       [1]           1          0         1         1         1         3
    3        2                 [1, 1, 0]           6          6         3         3         3         9
    5        3           [1, 1, 1, 0, 0]          35         20         5         5         5        15
    7        4     [1, 1, 1, 1, 0, 0, 0]         210         42         7         7         7        21
    9        5  [1, 1, 1, 1, 1, 0, 0, 0, 0]        1287         72         9         9         9        27
   11        6  [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]        8008        110        11        11        11        33
   13        7  [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]       50388        156        13        13        13        39
   15        8  [1, 1, 1, 1, 1, 1, 1,

In [10]:
def train_one_model(n_modes, n_photons, epochs_override=None, verbose=False, keep_model=False):
    """Train one model for a fixed number of modes and photons."""
    epochs_here = sweep_epochs if epochs_override is None else epochs_override

    # Re-seed each model for reproducibility of the sweep.
    local_seed = seed + 100 * n_photons + n_modes
    torch.manual_seed(local_seed)
    np.random.seed(local_seed)

    model = MerlinHeatQPINNCustom(
        n_modes=n_modes,
        n_photons=n_photons,
        hidden=hidden,
        mesh_depth=mesh_depth,
    ).to(device=device, dtype=dtype)

    # Defensive dtype alignment around the MerLin layer.
    for p in model.parameters():
        if p.is_floating_point():
            p.data = p.data.to(dtype)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []

    for epoch in range(1, epochs_here + 1):
        optimizer.zero_grad()

        xt_f = sample_interior(n_f)
        xt_i = sample_initial(n_i)
        xt_b = sample_boundary(n_b)

        # Physics and consistency losses
        r_f, r_c = pde_and_consistency_residuals(model, xt_f)
        loss_f = mse(r_f, torch.zeros_like(r_f))
        loss_c = mse(r_c, torch.zeros_like(r_c))

        # Initial-condition loss
        u_i, _ = model(xt_i)
        x_i = xt_i[:, 0:1]
        t_i = xt_i[:, 1:2]
        loss_i = mse(u_i, exact_u(x_i, t_i))

        # Boundary loss: should already be near zero by construction.
        u_b, _ = model(xt_b)
        loss_b = mse(u_b, torch.zeros_like(u_b))

        loss = (
            lambda_f * loss_f
            + lambda_c * loss_c
            + lambda_i * loss_i
            + lambda_b * loss_b
        )
        loss.backward()
        optimizer.step()

        history.append([
            loss.item(),
            loss_f.item(),
            loss_c.item(),
            loss_i.item(),
            loss_b.item(),
        ])

        if verbose and (epoch == 1 or epoch % 25 == 0):
            print(
                f"photons={n_photons} | modes={n_modes} | epoch {epoch:04d} | "
                f"loss={loss.item():.4e} | pde={loss_f.item():.2e} | "
                f"cons={loss_c.item():.2e} | ic={loss_i.item():.2e} | "
                f"bc={loss_b.item():.2e}"
            )

    hist = np.array(history)

    with torch.no_grad():
        u_pred, _ = model(xt_grid)
        u_true = exact_u(xt_grid[:, 0:1], xt_grid[:, 1:2])
        rel_l2 = (
            torch.linalg.norm(u_pred - u_true)
            / torch.linalg.norm(u_true)
        ).item()

    counts = gate_counts(n_modes, mesh_depth=mesh_depth)
    result = {
        "n_modes": n_modes,
        "n_photons": n_photons,
        "history": hist,
        "rel_l2": rel_l2,
        "final_total": hist[-1, 0],
        "final_pde": hist[-1, 1],
        "final_consistency": hist[-1, 2],
        "final_initial": hist[-1, 3],
        "final_boundary": hist[-1, 4],
        "quantum_output_size": model.quantum.output_size,
        "trainable_parameters": sum(
            p.numel() for p in model.parameters() if p.requires_grad
        ),
        **counts,
    }

    if keep_model:
        result["model"] = model
        result["U_true"] = u_true.reshape(nx, nt).detach().cpu()
        result["U_pred"] = u_pred.reshape(nx, nt).detach().cpu()
    else:
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return result

In [ ]:
# Sweep over the requested configurations.
# For each iteration, show the quantum circuit and print gate counts before training.
results = []

for n_modes, n_photons in config_list:
    counts = gate_counts(n_modes, mesh_depth=mesh_depth)
    circ = build_paper_inspired_circuit(
        n_modes=n_modes,
        n_photons=n_photons,
        mesh_depth=mesh_depth,
    )

    print("\n" + "=" * 90)
    print(f"Circuit for {n_modes} mode(s), {n_photons} photon(s)")
    print("input state:", balanced_photon_input_state(n_modes, n_photons))
    print("full-Fock output size:", fock_output_size(n_modes, n_photons))
    print(
        "gate counts | "
        f"BS(U1)={counts['BS_U1']} | BS(U2)={counts['BS_U2']} | "
        f"BS total={counts['BS_total']} | "
        f"PS input={counts['PS_input']} | PS theta={counts['PS_theta']} | "
        f"PS alpha={counts['PS_alpha']} | PS total={counts['PS_total']}"
    )
    display_circuit_inline(circ, f"circuit_{n_modes}modes_{n_photons}photons.svg")

    print(f"Training model with {n_photons} photon(s), {n_modes} mode(s)...")
    result = train_one_model(
        n_modes=n_modes,
        n_photons=n_photons,
        verbose=False,
        keep_model=False,
    )
    results.append(result)
    print(
        f"relative L2={result['rel_l2']:.4e} | "
        f"q_out={result['quantum_output_size']} | "
        f"params={result['trainable_parameters']}"
    )

best_sweep_result = min(results, key=lambda r: r["rel_l2"])
print("\nBest sweep configuration:")
print("photons:", best_sweep_result["n_photons"])
print("modes:", best_sweep_result["n_modes"])
print("relative L2 error:", best_sweep_result["rel_l2"])


Circuit for 1 mode(s), 1 photon(s)
input state: [1]
full-Fock output size: 1
gate counts | BS(U1)=0 | BS(U2)=0 | BS total=0 | PS input=1 | PS theta=1 | PS alpha=1 | PS total=3
SVG export failed ([Errno 2] No such file or directory: 'rendered_circuits/circuit_1modes_1photons.svg'); falling back to pcvl.pdisplay.
Training model with 1 photon(s), 1 mode(s)...
relative L2=3.3415e-01 | q_out=1 | params=69

Circuit for 3 mode(s), 2 photon(s)
input state: [1, 1, 0]
full-Fock output size: 6
gate counts | BS(U1)=3 | BS(U2)=3 | BS total=6 | PS input=3 | PS theta=3 | PS alpha=3 | PS total=9
SVG export failed ([Errno 2] No such file or directory: 'rendered_circuits/circuit_3modes_2photons.svg'); falling back to pcvl.pdisplay.
Training model with 2 photon(s), 3 mode(s)...
relative L2=1.6183e-01 | q_out=6 | params=143

Circuit for 5 mode(s), 3 photon(s)
input state: [1, 1, 1, 0, 0]
full-Fock output size: 35
gate counts | BS(U1)=10 | BS(U2)=10 | BS total=20 | PS input=5 | PS theta=5 | PS alpha=5 | P

In [ ]:
# Compact summary table
header = (
    f"{'modes':>5}  {'photons':>7}  {'q_out':>10}  {'params':>10}  {'rel_L2':>12}  "
    f"{'BS_total':>9}  {'PS_total':>8}  {'total':>12}  {'consistency':>12}  "
    f"{'initial':>12}  {'boundary':>12}"
)
print(header)
print("-" * len(header))
for r in results:
    print(
        f"{r['n_modes']:>5d}  {r['n_photons']:>7d}  "
        f"{r['quantum_output_size']:>10d}  {r['trainable_parameters']:>10d}  "
        f"{r['rel_l2']:>12.4e}  {r['BS_total']:>9d}  {r['PS_total']:>8d}  "
        f"{r['final_total']:>12.4e}  {r['final_consistency']:>12.4e}  "
        f"{r['final_initial']:>12.4e}  {r['final_boundary']:>12.4e}"
    )

In [ ]:
# Sweep plots.
# PDE loss is intentionally not plotted.
metrics = [
    ("relative L2 error", "rel_l2"),
    ("final total loss", "final_total"),
    ("final consistency loss", "final_consistency"),
    ("final initial loss", "final_initial"),
    ("final boundary loss", "final_boundary"),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

modes = np.array([r["n_modes"] for r in results])

for ax, (title, key) in zip(axes, metrics):
    values = np.array([r[key] for r in results], dtype=float)
    safe_values = np.maximum(values, 1e-20)
    ax.plot(modes, safe_values, marker="o")
    ax.set_yscale("log")
    ax.set_xlabel("number of modes")
    ax.set_ylabel(title)
    ax.set_xticks(mode_list)
    ax.grid(True, which="both", alpha=0.3)
    ax.set_title(title)

axes[-1].axis("off")
fig.suptitle("Mode/photon sweep: photons = (modes + 1) / 2", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Retrain the best sweep configuration for a longer run so the displayed
# curves are comparable to the earlier 300-epoch notebooks.
print(
    f"Retraining best configuration for {best_retrain_epochs} epochs: "
    f"{best_sweep_result['n_photons']} photon(s), "
    f"{best_sweep_result['n_modes']} mode(s)"
)
best_result = train_one_model(
    n_modes=best_sweep_result["n_modes"],
    n_photons=best_sweep_result["n_photons"],
    epochs_override=best_retrain_epochs,
    verbose=False,
    keep_model=True,
)

print("Retrained best relative L2 error:", best_result["rel_l2"])
print("Quantum output size:", best_result["quantum_output_size"])
print("Trainable parameters:", best_result["trainable_parameters"])

In [ ]:
# Display the best circuit from the sweep and its gate counts
print(
    f"Best circuit: {best_result['n_photons']} photon(s), "
    f"{best_result['n_modes']} mode(s)"
)
print(
    "gate counts | "
    f"BS(U1)={best_result['BS_U1']} | BS(U2)={best_result['BS_U2']} | "
    f"BS total={best_result['BS_total']} | "
    f"PS input={best_result['PS_input']} | PS theta={best_result['PS_theta']} | "
    f"PS alpha={best_result['PS_alpha']} | PS total={best_result['PS_total']}"
)
display_circuit_inline(
    best_result["model"].circuit_to_draw,
    f"best_circuit_{best_result['n_modes']}modes_{best_result['n_photons']}photons.svg",
)

In [ ]:
# Training curves for the circuit with the minimum sweep relative L2 error.
# PDE loss is intentionally not shown.
best_hist = best_result["history"]
plt.figure(figsize=(7, 4))
plt.semilogy(best_hist[:, 0], label="total")
plt.semilogy(best_hist[:, 2], label="consistency")
plt.semilogy(best_hist[:, 3], label="initial")
plt.semilogy(best_hist[:, 4], label="boundary")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.title(
    f"Best model: {best_result['n_photons']} photon(s), "
    f"{best_result['n_modes']} mode(s), "
    f"relative L2={best_result['rel_l2']:.3e}"
)
plt.show()

In [ ]:
# Visual comparison for the retrained best model
U_true = best_result["U_true"]
U_pred = best_result["U_pred"]

plt.figure(figsize=(6, 5))
plt.imshow(U_true.numpy(), origin="lower", extent=[0, 1, 0, 1], aspect="auto")
plt.colorbar(label="u")
plt.xlabel("t")
plt.ylabel("x")
plt.title("Exact heat equation solution")
plt.show()

plt.figure(figsize=(6, 5))
plt.imshow(U_pred.numpy(), origin="lower", extent=[0, 1, 0, 1], aspect="auto")
plt.colorbar(label="u")
plt.xlabel("t")
plt.ylabel("x")
plt.title("Best MerLin DV-photonic QPINN prediction")
plt.show()

plt.figure(figsize=(6, 5))
plt.imshow((U_pred - U_true).abs().numpy(), origin="lower", extent=[0, 1, 0, 1], aspect="auto")
plt.colorbar(label="absolute error")
plt.xlabel("t")
plt.ylabel("x")
plt.title("Absolute error")
plt.show()